# Relevance@5 QA Validation

## 1. Purpose

Relevance@5 measures the fraction of the top effective-K retrieved documents that are relevant to the query. When fewer than five documents are supplied, effective-K is the supplied count.


## 2. Imports and output location

The helper locates the repository root whether Jupyter starts at the repository root or inside `notebooks/qa`.


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from idp_eval import EvaluationCase, EvaluationFramework, create_azure_judge
from idp_eval.judges import AzureJudgeConfig

from idp_eval import RelevanceAtKEvaluator


In [ ]:
def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "idp_eval").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from within the idp-eval repository.")


REPO_ROOT = find_repo_root()
OUTPUT_DIR = REPO_ROOT / "qa_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 3. Configure the judge

Replace every placeholder before running. No Phoenix server is required. If your
application already constructs a compatible judge, you may replace this cell
with that existing construction. Keep credentials in your application's secret
management system rather than saving them in this notebook.


In [ ]:
azure_config = AzureJudgeConfig(
    model="YOUR_AZURE_DEPLOYMENT",
    azure_endpoint="YOUR_AZURE_ENDPOINT",
    tenant_id="YOUR_TENANT_ID",
    client_id="YOUR_CLIENT_ID",
    client_secret="YOUR_CLIENT_SECRET",
    api_version="2024-12-01-preview",
    timeout=180,
    proxy_url=None,
    verify_ssl=True,
    reasoning_effort=None,
)

judge = create_azure_judge(config=azure_config)


## 4. Five mock evaluation cases

These cases intentionally span clear pass, partial, and fail behaviors. `expected_behavior` is a human QA aid, not an exact model-score assertion.


In [ ]:
cases = [
    EvaluationCase(
        case_id="REL-001",
        input="How can an employee reset a forgotten password?",
        retrieved_documents=[
            {"document_id": "pw-1", "text": "Reset a forgotten password from the sign-in page.", "score": 0.99},
            {"document_id": "pw-2", "text": "The password reset email expires after 15 minutes.", "score": 0.95},
            {"document_id": "pw-3", "text": "Contact the help desk if password reset fails.", "score": 0.91},
            {"document_id": "pw-4", "text": "MFA verification is required during password reset.", "score": 0.89},
            {"document_id": "pw-5", "text": "Choose a new password with at least 12 characters.", "score": 0.86},
        ],
    ),
    EvaluationCase(
        case_id="REL-002",
        input="How do customers pay an invoice?",
        retrieved_documents=[
            {"document_id": "inv-1", "text": "Invoices can be paid by card or ACH from the billing portal.", "score": 0.98},
            {"document_id": "vac-1", "text": "Employees receive 20 vacation days each year.", "score": 0.94},
            {"document_id": "inv-2", "text": "Use the invoice number when submitting a bank payment.", "score": 0.90},
            {"document_id": "ship-1", "text": "Customers can update a shipping address before dispatch.", "score": 0.87},
            {"document_id": "inv-3", "text": "The billing portal displays payment confirmation.", "score": 0.84},
        ],
    ),
    EvaluationCase(
        case_id="REL-003",
        input="How can a customer change a shipping address?",
        retrieved_documents=[
            {"document_id": "db-1", "text": "Database backups run every night.", "score": 0.97},
            {"document_id": "inv-4", "text": "Overdue invoices receive a reminder email.", "score": 0.93},
            {"document_id": "ship-2", "text": "Change the shipping address on the order before it is dispatched.", "score": 0.90},
            {"document_id": "vac-2", "text": "Vacation requests require manager approval.", "score": 0.86},
            {"document_id": "pw-6", "text": "Password reset links expire after 15 minutes.", "score": 0.81},
        ],
    ),
    EvaluationCase(
        case_id="REL-004",
        input="What is the employee vacation policy?",
        retrieved_documents=[
            {"document_id": "db-2", "text": "Database backups are retained for 30 days.", "score": 0.96},
            {"document_id": "ship-3", "text": "Shipping addresses can be changed before dispatch.", "score": 0.92},
            {"document_id": "inv-5", "text": "Invoice payments accept card and ACH.", "score": 0.88},
            {"document_id": "pw-7", "text": "Passwords require at least 12 characters.", "score": 0.84},
            {"document_id": "db-3", "text": "Restore tests run each quarter.", "score": 0.80},
        ],
    ),
    EvaluationCase(
        case_id="REL-005",
        input="How are database backups managed?",
        retrieved_documents=[
            {"document_id": "db-4", "text": "Production databases are backed up nightly.", "score": 0.99},
            {"document_id": "db-5", "text": "Backups are retained for 30 days.", "score": 0.95},
            {"document_id": "db-6", "text": "Restore tests validate backups every quarter.", "score": 0.90},
        ],
    ),
]

expected_behavior = {
    "REL-001": "all_relevant — approximately 5/5",
    "REL-002": "partially_relevant — approximately 3/5",
    "REL-003": "partially_relevant — approximately 1/5",
    "REL-004": "none_relevant — approximately 0/5",
    "REL-005": "all_relevant with effective_k=3 although requested k=5",
}


## 5. Inspect the mock inputs


In [ ]:
case_rows = []
for case in cases:
    case_rows.append({
        "case_id": case.case_id,
        "expected_behavior": expected_behavior[case.case_id],
        "input": getattr(case, "input"),
        "retrieved_documents": getattr(case, "retrieved_documents"),
    })

cases_df = pd.DataFrame(case_rows)
display(cases_df)


## 6. Configure one evaluator and Excel output

This notebook runs exactly one metric. `resume=False` creates a fresh QA workbook and no Phoenix tracing is configured.


In [ ]:
excel_path = OUTPUT_DIR / "relevance_at_k_validation.xlsx"
evaluator = RelevanceAtKEvaluator(k=5, verbose=True)
framework = EvaluationFramework(
    evaluators=[evaluator],
    judge=judge,
    output="excel",
    excel_path=str(excel_path),
    resume=False,
)

results = framework.evaluate_many(
    cases,
    run_name="qa-validation",
    dataset_name="mock-acceptance-cases",
    show_progress=True,
)


## 7. Result summary


In [ ]:
METRIC_NAME = "relevance_at_5"
summary_rows = []
for case, result_map in zip(cases, results, strict=True):
    result = result_map[METRIC_NAME]
    summary_rows.append({
        "case_id": case.case_id,
        "expected_behavior": expected_behavior[case.case_id],
        "score": result.score,
        "label": result.label,
        "explanation": result.explanation,
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)


## 8. Inspect Excel output

The workbook summary is in `evaluations`; item-level evidence is in `retrieval_documents`.


In [ ]:
evaluations_df = pd.read_excel(excel_path, sheet_name="evaluations")
display(evaluations_df)

details_df = pd.read_excel(excel_path, sheet_name="retrieval_documents")

visible_columns = [
    column
    for column in (
        "key_id",
        "rank",
        "document_id",
        "text",
        "relevant",
        "relevance_score",
        "reason",
        "retrieval_score",
    )
    if column in details_df.columns
]
display(details_df[visible_columns])


## 9. Sanity assertions

These assertions validate framework/output behavior and broad direction only; they do not require exact LLM-generated fractions.


In [ ]:
assert len(results) == 5
assert excel_path.exists()
assert all(METRIC_NAME in result_map for result_map in results)
assert len(evaluations_df) == 5
assert set(evaluations_df["key_id"]) == {case.case_id for case in cases}
assert set(("text", "relevant", "relevance_score", "reason")).issubset(details_df.columns)
assert results[0][METRIC_NAME].score >= results[3][METRIC_NAME].score
assert results[4][METRIC_NAME].details["effective_k"] == 3
print("Relevance@5 QA sanity checks passed.")


## 10. Close judge resources and report the workbook path


In [ ]:
judge.close()
print(f"Excel output: {excel_path.resolve()}")
